# RAISE: End-to-End Annual Report PDF Parsing & HTML5 Semantification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semanticClimate/RAISE/blob/siddharth-semantification/RAISE_PDF_Parsing_and_Semantification_Colab.ipynb)

This notebook unifies **Stage 1 (Layout-Aware PDF Block Extraction)**, **Stage 2 (Noise Filtering & Target Block Identification)**, and **Stage 3 (Semantic HTML5 Document Generation & Agentic AI Chunking)** into a single end-to-end pipeline for **Agentic AI**.

**Dependencies**: Fetches engine modules from PyPI and GitHub branches (`Harsh-semantification` and `siddharth-semantification`).

### Pipeline Architecture:
1. **Stage 1 (PDF Parsing)**: Extract layout-aware blocks, headings, paragraphs, lists, and tables using `pymupdf` & `pdfplumber`.
2. **Stage 2 (Noise Filtering)**: Filter out non-research pages (financial audit/balance sheets, photo galleries, staff directories, administrative notices, and running headers/footers) to produce high-density `target_blocks.json`.
3. **Stage 3 (HTML5 Semantification)**: Validate JSON schema input quality, dynamically learn structural font profiles, construct nested section hierarchy trees, render traceable semantic HTML5 (`report.html`), and generate AI-ready LLM RAG chunks (`ai_chunks.json`) and source mappings.

---


## Step 1: Upload the Annual Report PDF File

Upload your Annual Report PDF file below to execute the RAISE semantification pipeline.


In [ ]:
from pathlib import Path

input_pdf_filename = None

try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    print("Upload your Annual Report PDF file below:")
    uploaded = files.upload()
    if uploaded:
        filename, file_bytes = next(iter(uploaded.items()))
        if filename.lower().endswith(".pdf"):
            input_pdf_filename = filename
            print(f"Uploaded PDF file successfully: {input_pdf_filename} ({len(file_bytes)} bytes)")
        else:
            raise ValueError(f"File '{filename}' is not a .pdf file. Please upload a valid PDF document.")
    else:
        raise FileNotFoundError("No file uploaded. Please upload an Annual Report PDF file to proceed.")
else:
    # Local environment / automated execution
    input_pdf_filename = "sample_annual_report.pdf"
    if not Path(input_pdf_filename).exists():
        import pymupdf as fitz
        doc = fitz.open()
        p1 = doc.new_page()
        p1.insert_text((72, 72), "DEPARTMENT OF COMPUTER SCIENCE & AI", fontsize=20, fontname="hebo")
        p1.insert_text((72, 110), "Annual Report 2025-2026", fontsize=16, fontname="hebo")
        p1.insert_text((72, 150), "1. Research Highlights", fontsize=14, fontname="hebo")
        p1.insert_text((72, 180), "Our department pioneered research in agentic AI, automated scientific semantic markup, and LLM structured knowledge representation.", fontsize=11, fontname="tiro")
        p1.insert_text((72, 220), "Key publications include articles in top peer-reviewed journals and conference proceedings.", fontsize=11, fontname="tiro")
        doc.save(input_pdf_filename)
        doc.close()

print(f"Active PDF file for pipeline execution: {input_pdf_filename}")


## Step 2: Upload an Optional Custom Content Schema

Set `UPLOAD_CUSTOM_SCHEMA` to `True` if you wish to upload a custom JSON Schema for domain-specific block filtering.


In [ ]:
import json

UPLOAD_CUSTOM_SCHEMA = False  # @param {type:"boolean"}
custom_schema = None

if UPLOAD_CUSTOM_SCHEMA and is_colab:
    print("Choose one JSON Schema file (.json):")
    try:
        schema_upload = files.upload()
        if len(schema_upload) == 1:
            schema_filename, schema_bytes = next(iter(schema_upload.items()))
            custom_schema = json.loads(schema_bytes.decode("utf-8"))
            print(f"Loaded custom schema: {schema_filename}")
    except Exception as err:
        print(f"Schema upload skipped or cancelled ({err}).")
else:
    print("Custom JSON schema filtering disabled (using built-in RAISE filters).")


## Step 3: Install Dependencies & Fetch Engine Modules

Installs required dependencies from PyPI and clones the `semanticClimate/RAISE` repository online.


In [ ]:
# Pre-install dependencies, build backend, and visualization packages
!pip install -q pymupdf pdfplumber pydantic typer rich jsonschema beautifulsoup4 jinja2 lxml html5lib hatchling ipywidgets matplotlib seaborn

import os

# Clean reset of previous clone directory to prevent git status 128 errors
!rm -rf /content/RAISE_SRC ./RAISE_SRC

# Clone repository from public GitHub repo
print("Cloning RAISE repository from GitHub...")
!git clone -q https://github.com/semanticClimate/RAISE.git ./RAISE_SRC

print("Installing raise_pdf_parsing from Harsh-semantification branch...")
!git -C ./RAISE_SRC fetch -q origin Harsh-semantification
!git -C ./RAISE_SRC checkout -B Harsh-semantification origin/Harsh-semantification -q
!pip install -q ./RAISE_SRC/RAISE_PDF_Parsing

print("Installing raise_html5_semantification from siddharth-semantification branch...")
!git -C ./RAISE_SRC fetch -q origin siddharth-semantification
!git -C ./RAISE_SRC checkout -B siddharth-semantification origin/siddharth-semantification -q
!pip install -q ./RAISE_SRC/RAISE_HTML5_Semantification

print("Both RAISE monorepo engine modules installed successfully!")


## Step 4: Import Engines & Execute Verification Smoke Tests

Imports `raise_pdf_parsing` and `raise_html5_semantification` and runs synthetic in-memory smoke tests to verify runtime integrity.


In [ ]:
import io
import json
from pathlib import Path
from tempfile import TemporaryDirectory
import pymupdf as fitz
from PIL import Image

import raise_pdf_parsing
from raise_pdf_parsing.filter import filter_blocks
from raise_pdf_parsing.main import run_pipeline
from raise_pdf_parsing.parser import parse_pdf
from raise_pdf_parsing.schema_filter import filter_target_by_schema, validate_custom_schema

import raise_html5_semantification
from raise_html5_semantification.html_writer import render_document
from raise_html5_semantification.loader import load_blocks
from raise_html5_semantification.main import build_report
from raise_html5_semantification.profiler import learn_semantic_profile
from raise_html5_semantification.section_builder import build_section_tree
from raise_html5_semantification.validator import validate_html_string

print(f"Loaded raise_pdf_parsing v{raise_pdf_parsing.__version__}")
print(f"Loaded raise_html5_semantification engine from: {Path(raise_html5_semantification.__file__).resolve()}")

# --- Stage 1 & 2 Synthetic PDF Filter Smoke Test ---
def _square(color, size=220):
    buf = io.BytesIO()
    Image.new("RGB", (size, size), color=color).save(buf, format="PNG")
    return buf.getvalue()

_doc = fitz.open()
_p = _doc.new_page()
_p.insert_text((72, 110), "RESEARCH ACTIVITIES", fontsize=18, fontname="hebo")
_p.insert_text((72, 150), "Funded research continued across departments.", fontsize=11, fontname="tiro")
_p = _doc.new_page()
_p.insert_text((72, 110), "STATEMENT OF ACCOUNTS", fontsize=16, fontname="hebo")
_p.insert_text((72, 150), "Balance Sheet and Auditor's Report attached, trial balance reconciled.", fontsize=11, fontname="tiro")
_p = _doc.new_page()
for _x, _y in [(72, 140), (320, 140), (72, 420), (320, 420)]:
    _p.insert_image(fitz.Rect(_x, _y, _x + 220, _y + 220), stream=_square((200, 200, 200)))
_smoke_pdf = Path("_smoke_test_report.pdf")
_doc.save(_smoke_pdf)
_doc.close()

_smoke_report = parse_pdf(_smoke_pdf)
_smoke_target = filter_blocks(_smoke_report)
_smoke_meta = _smoke_target.filtering_metadata
print(f"PDF Parsing Smoke test passed: {_smoke_meta.output_block_count}/{_smoke_meta.input_block_count} blocks kept.")

# --- Stage 3 Synthetic HTML5 Semantification Smoke Test ---
SMOKE_CASES = {
    "minimal_missing_ids": [
        {"text": "PUBLICATIONS", "font_size": 19, "is_bold": True, "confidence": 0.96},
        {"text": "The faculty published journal articles and conference papers.", "confidence": 0.85},
        {"text": "- Journal articles: 45\n- Conference papers: 62", "confidence": 0.82},
    ],
    "nested_annual_report": {
        "document": {
            "pages": [
                {
                    "pageNumber": 4,
                    "elements": [
                        {"content": "DEPARTMENT OF COMPUTER SCIENCE", "page": 4, "fontSize": 20, "bold": True, "role": "heading", "probability": 0.99},
                        {"content": "Research Grants and Collaboration", "page": 4, "fontSize": 15, "bold": True, "section": "Grants", "probability": 0.94},
                        {"content": "1. DST project continued\n2. Industry collaboration signed", "page": 4, "probability": 0.91},
                    ],
                }
            ]
        }
    },
}

def semantify_raw_for_smoke(raw, name):
    with TemporaryDirectory() as temp_dir:
        input_path = Path(temp_dir) / f"{name}.json"
        input_path.write_text(json.dumps(raw), encoding="utf-8")
        blocks = load_blocks(input_path)
        profile = learn_semantic_profile(blocks, source_name=name)
        nodes = build_section_tree(blocks, profile=profile)
        report_html = render_document(nodes, title=f"Smoke test: {name}")
        summary = validate_html_string(report_html, expected_blocks=len(blocks))
        return summary.ok

html_smoke_ok = all(semantify_raw_for_smoke(raw, name) for name, raw in SMOKE_CASES.items())
assert html_smoke_ok, "HTML5 Semantification smoke test failed."
print("HTML5 Semantification Smoke test passed successfully!")


## Step 5: Execute Stage 1 & 2 -- PDF Layout Parsing & Noise Filtering

Parses the PDF into layout blocks (Stage 1) and excludes non-research noise pages (Stage 2) to produce `target_blocks.json`. Displays Python visual analytics charts.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

parsed_path = "report_blocks.json"
target_path = "target_blocks.json"
filtering_report_path = "filtering_report.json"
schema_filter_report_path = "schema_filtering_report.json"

print(f"Running Stage 1 & Stage 2 pipeline on '{input_pdf_filename}'...")
parsed, target = run_pipeline(
    pdf_path=input_pdf_filename,
    parsed_output_path=parsed_path,
    target_output_path=target_path,
    filtering_report_path=filtering_report_path,
)

if custom_schema is not None:
    validate_custom_schema(custom_schema)
    target, schema_filter_report = filter_target_by_schema(target, custom_schema)
    target_payload = target.model_dump(exclude_none=False)
    TargetBlocksDocument.model_validate(target_payload)
    Path(target_path).write_text(json.dumps(target_payload, indent=2, ensure_ascii=False), encoding="utf-8")
    Path(schema_filter_report_path).write_text(json.dumps(schema_filter_report, indent=2, ensure_ascii=False), encoding="utf-8")

meta = target.filtering_metadata
retained_count = meta.output_block_count
total_count = meta.input_block_count
excluded_pages_count = len(meta.excluded_pages)
retained_pages_count = len(meta.retained_pages)

print("\n--- STAGE 1 & 2 SUMMARY ---")
print(f"Parsed Pages: {parsed.page_count} pages | Raw Blocks: {total_count} blocks")
print(f"Retained Blocks: {retained_count} blocks across {retained_pages_count} pages")
print(f"Excluded Pages: {excluded_pages_count} page(s)")
for record in meta.excluded_pages:
    print(f"  - Page {record.page_number}: {record.category} ({record.reason})")
print(f"Removed Boilerplate Blocks: {meta.removed_boilerplate_block_count}")

# --- Matplotlib Dark-Theme Pipeline Visualization ---
plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('#0f172a')
ax1.set_facecolor('#1e293b')
ax2.set_facecolor('#1e293b')

# Chart 1: Retained vs Excluded Page Breakdown
page_labels = ['Retained Research Pages', 'Excluded Noise Pages']
page_sizes = [max(retained_pages_count, 1), excluded_pages_count]
colors = ['#10b981', '#f59e0b']
wedges, texts, autotexts = ax1.pie(page_sizes, labels=page_labels, colors=colors, autopct='%1.1f%%', startangle=140, textprops=dict(color='w', weight='bold'))
ax1.set_title('Stage 2 Page Filter Breakdown', fontsize=13, fontweight='bold', color='#f8fafc', pad=12)

# Chart 2: Block Density Retained
categories = ['Raw Extracted', 'Retained Research']
values = [total_count, retained_count]
bars = ax2.bar(categories, values, color=['#a855f7', '#10b981'], width=0.5)
ax2.set_ylabel('Number of Blocks', color='#94a3b8', fontweight='bold')
ax2.set_title('Stage 1 vs Stage 2 Density', fontsize=13, fontweight='bold', color='#f8fafc', pad=12)
ax2.grid(axis='y', linestyle='--', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1, f'{int(height)}', ha='center', va='bottom', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


## Step 6: Execute Stage 3 -- HTML5 Semantification & Agentic AI Chunking

Transforms `target_blocks.json` into traceable HTML5 (`report.html`) and extracts structured AI-ready chunks (`ai_chunks.json`) for LLM RAG pipelines.


In [ ]:
semantic_html_path = "report.html"
source_map_path = "source_map.json"
section_map_path = "section_map.json"
ai_chunks_path = "ai_chunks.json"
validation_report_path = "validation_report.json"
input_quality_report_path = "input_quality_report.json"
semantic_profile_path = "semantic_profile.json"

print("Running Stage 3 HTML5 Semantification on 'target_blocks.json'...")
report_html = build_report(
    input_path=target_path,
    output_path=semantic_html_path,
    validation_output_path=validation_report_path,
    profile_output_path=semantic_profile_path,
    source_map_output_path=source_map_path,
    section_map_output_path=section_map_path,
    ai_chunks_output_path=ai_chunks_path,
    input_quality_output_path=input_quality_report_path,
)

validation_report = json.loads(Path(validation_report_path).read_text(encoding="utf-8"))
input_quality_report = json.loads(Path(input_quality_report_path).read_text(encoding="utf-8"))
semantic_profile = json.loads(Path(semantic_profile_path).read_text(encoding="utf-8"))
ai_chunks = json.loads(Path(ai_chunks_path).read_text(encoding="utf-8"))

print("\n--- STAGE 3 SEMANTIFICATION SUMMARY ---")
print(f"HTML5 File: {semantic_html_path} ({len(report_html)} characters)")
print(f"Validation Passed: {validation_report.get('ok', False)}")
print(f"Traceable Elements: {validation_report.get('traceable_element_count', 0)}")
print(f"AI Chunks Generated: {len(ai_chunks)} chunk(s) for Agentic AI consumption")


## Step 7: Review Audit Reports & Preview Generated HTML5

Prints quality metrics and renders the generated `report.html` natively inside an inline scrollable container.


In [ ]:
from IPython.display import HTML, display

print("=== Validation Audit Report ===")
print(json.dumps(validation_report, indent=2))

print("\n=== Input Quality Report ===")
print(json.dumps(input_quality_report, indent=2))

print("\n=== Learned Semantic Profile ===")
print(json.dumps(semantic_profile, indent=2))

print("\n=== Document Preview: report.html ===")
html_content = Path(semantic_html_path).read_text(encoding="utf-8")
# Native inline HTML container
display(HTML(f'<div style="max-height: 550px; overflow-y: auto; border: 2px solid #6366f1; border-radius: 8px; padding: 12px; background: #ffffff; color: #000000;">{html_content}</div>'))


## Step 8: Package & Interactive Selective File Downloader

Packages deliverables into `raise_end_to_end_outputs.zip` and presents an **interactive checklist & download selector** allowing you to select and download specific files.


In [ ]:
import base64
from zipfile import ZIP_DEFLATED, ZipFile
from IPython.display import HTML, display

file_registry = {
    "raise_end_to_end_outputs.zip": "Complete Deliverable Package (.zip)",
    "report.html": "Semantified HTML5 Report (report.html)",
    "ai_chunks.json": "Agentic AI & RAG Chunks (ai_chunks.json)",
    "target_blocks.json": "Filtered Target Blocks (target_blocks.json)",
    "source_map.json": "Source Traceability Map (source_map.json)",
    "section_map.json": "Document Outline Map (section_map.json)",
    "validation_report.json": "Validation Audit Report (validation_report.json)",
    "input_quality_report.json": "Input Quality Audit (input_quality_report.json)",
    "semantic_profile.json": "Learned Semantic Profile (semantic_profile.json)",
    "report_blocks.json": "Stage 1 Raw Layout Blocks (report_blocks.json)",
    "filtering_report.json": "Stage 2 Exclusions Report (filtering_report.json)",
}
if custom_schema is not None:
    file_registry["schema_filtering_report.json"] = "Custom Schema Report (schema_filtering_report.json)"

# Package zip file
full_zip_path = "raise_end_to_end_outputs.zip"
with ZipFile(full_zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for fname in file_registry.keys():
        if Path(fname).exists() and fname != full_zip_path:
            archive.write(fname, arcname=fname)

print(f"Packaged deliverable zip archive: {full_zip_path}")

# Render Interactive Download Checklist UI
html_items = []
for fname, label in file_registry.items():
    fpath = Path(fname)
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        content_b64 = base64.b64encode(fpath.read_bytes()).decode('utf-8')
        html_items.append(f'''
        <div style="display: flex; align-items: center; justify-content: space-between; padding: 10px 14px; margin-bottom: 8px; background: rgba(30, 41, 59, 0.7); border: 1px solid rgba(255, 255, 255, 0.1); border-radius: 8px;">
            <div style="display: flex; align-items: center; gap: 10px;">
                <input type="checkbox" id="check_{fname}" value="{fname}" checked style="width: 18px; height: 18px; cursor: pointer;">
                <label for="check_{fname}" style="color: #f8fafc; font-weight: 500; font-size: 14px; cursor: pointer;">{label} <span style="color: #94a3b8; font-size: 12px;">({size_kb:.1f} KB)</span></label>
            </div>
            <a href="data:application/octet-stream;base64,{content_b64}" download="{fname}" style="background: #4f46e5; color: #ffffff; padding: 6px 14px; text-decoration: none; border-radius: 6px; font-size: 13px; font-weight: 600; transition: background 0.2s;">Download</a>
        </div>
        ''')

download_selector_html = f'''
<div style="font-family: system-ui, -apple-system, sans-serif; background: #0f172a; border: 2px solid #6366f1; border-radius: 14px; padding: 20px; max-width: 750px; margin-top: 15px;">
    <h3 style="color: #f8fafc; margin-top: 0; font-size: 18px;">📥 Interactive Output File Selector</h3>
    <p style="color: #94a3b8; font-size: 13px; margin-bottom: 16px;">Select specific output files below to download individually or click download on any file:</p>
    {''.join(html_items)}
    <div style="margin-top: 16px; display: flex; gap: 12px;">
        <button onclick="downloadSelectedFiles()" style="background: linear-gradient(135deg, #10b981, #059669); color: white; border: none; padding: 10px 20px; border-radius: 8px; font-weight: 600; cursor: pointer; font-size: 14px;">⚡ Download Selected Checked Files</button>
    </div>
    <script>
    function downloadSelectedFiles() {{
        const checkboxes = document.querySelectorAll('input[type="checkbox"]:checked');
        checkboxes.forEach(cb => {{
            const link = cb.closest('div').querySelector('a');
            if (link) {{
                link.click();
            }}
        }});
    }}
    </script>
</div>
'''

display(HTML(download_selector_html))
